In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np

from prtlab import (
    add_surface,
    calc_k_from_theta_x_theta_y,
    create_optical_system,
    plot_polarization_ellipses_across_field,
    plot_prt_lens_cross_section,
    plot_prt_ray_trace,
    polarization_ray_trace,
    transform_p_to_jones,
    update_clear_apertures_from_ray_trace,
)


Example - Quarter Wave Plate
The simplest example of using prtLab is for a waveplate.  This example is inspired by Example 20.1 in PLAOS.

To start with, we need to define the optical system.  This is implemented using the OpticalSystem type in Python.  

In [ ]:
wavelength = 0.633  # um
n_o = 1.656
n_e = 1.485
t_qwp = 1.25 * wavelength / (n_o - n_e)
optic_axis = np.array([-np.sqrt(2) / 2, -np.sqrt(2) / 2, 0])

T = create_optical_system(wavelength)
T.wavelength_units = "um"

air_index = {"n": 1.0}
calcite_index = {"nO": n_o, "nE": n_e}
calcite_axis = {"opticAxis": optic_axis}
bare_coating = {"type": "bare"}

# Object/input space. The zero-thickness row makes the incident
# medium explicit without adding propagation.
add_surface(
    T, np.inf, 0, "plane", {}, "isotropic",
    air_index, {}, bare_coating,
)

# Plane entrance surface followed by propagation through the
# calcite plate.
add_surface(
    T, np.inf, t_qwp, "plane", {}, "uniaxial",
    calcite_index, calcite_axis, bare_coating,
)

# Plane exit surface into air. Thickness is zero because this row
# describes the output medium after the waveplate.
add_surface(
    T, np.inf, 0, "plane", {}, "isotropic",
    air_index, {}, bare_coating,
)


This gives a feel on how things are defined.  Materials can either be isotropic or uniaxial, and for uniaxial materials the axis of the extraordinary index needs to be defined as cos vectors.

We can see the output by looking at T.

In [ ]:
T


You might wonder why to have a o thickness isotropic material here.  The reason is so we can properly add results from nonzero angles of incidence, as we will do later. 

Now let's do a polarization ray trace.  The interface has a few arguments.  We need to specify the optical system (T), the incident k vector and position vector, and the incident E-field

In [ ]:
k_in = np.array([0, 0, 1])  # normal incidence
pos_in = np.array([0, 0, 0])  # on axis
E_in = np.array([0, 1])  # can be 2D or 3D vector
trace_options = {"encodePropagationPhaseInP": True}
ray_output = polarization_ray_trace(
    T, k_in, pos_in, E_in, trace_options
)


Let's look at the output:

In [ ]:
ray_output


You can see that there are 5 rays and two finalRayIds.  This is because the uniaxial material generates multiple rays.  There is a tree structure to keep track of things (as suggested in Chapter 19 of PLAOS).

Now we can use the result.  The first thing is to verify it actually produces the correct retardance based on the setup.  This can be done by simply accessing the OPL property for the last two rays.

In [ ]:
first, second = ray_output.final_ray_ids[:2]
opl = (
    ray_output.rays[first - 1].opl
    - ray_output.rays[second - 1].opl
)
opl_in_waves = opl / wavelength
opl_in_waves


So this basic test worked.  Now for something slighly more interesting, let's look at the angle of incidence dependence.  Since this is a true 0 order plate, it shouldn't be too sensitive.

In [ ]:
angles = np.linspace(0, 5, 21)
opl_vs_aoi = np.zeros_like(angles)

for index, angle in enumerate(angles):
    theta = np.deg2rad(angle)
    k_in = np.array([0, np.sin(theta), np.cos(theta)])
    ray_output = polarization_ray_trace(
        T, k_in, pos_in, E_in, trace_options
    )
    first, second = ray_output.final_ray_ids[:2]
    opl = (
        ray_output.rays[first - 1].opl
        - ray_output.rays[second - 1].opl
    )
    opl_vs_aoi[index] = opl / wavelength

plt.figure()
_ = plt.plot(angles, opl_vs_aoi - 1)
plt.xlabel("Angle of Incidence [deg]")
_ = plt.ylabel("Retardance [waves]")


As expected, sensitivity is low.  But making a < 5um thick waveplate is not that practical, so let's look at a multi order waveplate. 

In [ ]:
Tm = copy.deepcopy(T)
# Integer multiple guarantees quarter wave at 0 AOI.
Tm.surfaces[1].thickness = 17 * T.surfaces[1].thickness
opl_multiwave_vs_aoi = np.zeros_like(angles)

for index, angle in enumerate(angles):
    theta = np.deg2rad(angle)
    k_in = np.array([0, np.sin(theta), np.cos(theta)])
    ray_output = polarization_ray_trace(
        Tm, k_in, pos_in, E_in, trace_options
    )
    first, second = ray_output.final_ray_ids[:2]
    opl = (
        ray_output.rays[first - 1].opl
        - ray_output.rays[second - 1].opl
    )
    opl_multiwave_vs_aoi[index] = opl / wavelength

plt.figure()
_ = plt.plot(angles, opl_multiwave_vs_aoi - 21)
plt.xlabel("Angle of Incidence [deg]")
_ = plt.ylabel("Retardance [waves]")


The sensitivity is much higher.  As a more visual example, let's look at the situation where the waveplates are placed in a pupil plane with this angular range

In [ ]:
X, Y = np.meshgrid(np.linspace(-5, 5, 17), np.linspace(-5, 5, 17))
tX, tY = np.meshgrid(np.linspace(-5, 5, 17), np.linspace(-5, 5, 17))
coordinate = {
    "type": "doublePole",
    "a_loc": np.array([0, 0, 1]),
    "x_o": np.array([1, 0, 0]),
}
Jall = np.zeros((*X.shape, 2, 2), dtype=complex)
Jall_m = np.zeros_like(Jall)

for index in np.ndindex(X.shape):
    k_in = calc_k_from_theta_x_theta_y(tX[index], tY[index])
    pos_in = np.array([X[index], Y[index], 0])
    ray_output = polarization_ray_trace(
        T, k_in, pos_in, E_in, trace_options
    )
    P_tot = sum(
        (ray_output.rays[ray_id - 1].P
         for ray_id in ray_output.final_ray_ids),
        np.zeros((3, 3), dtype=complex),
    )
    k_out = ray_output.rays[ray_output.final_ray_ids[0] - 1].k
    Jall[index] = transform_p_to_jones(
        P_tot, k_in, k_out, coordinate
    )

    ray_output_m = polarization_ray_trace(
        Tm, k_in, pos_in, E_in, trace_options
    )
    P_tot_m = sum(
        (ray_output_m.rays[ray_id - 1].P
         for ray_id in ray_output_m.final_ray_ids),
        np.zeros((3, 3), dtype=complex),
    )
    k_out_m = ray_output_m.rays[
        ray_output_m.final_ray_ids[0] - 1
    ].k
    Jall_m[index] = transform_p_to_jones(
        P_tot_m, k_in, k_out_m, coordinate
    )

axis, _ = plot_polarization_ellipses_across_field(
    X, Y, Jall, E_in
)
_ = axis.set_title("True 0 Order Plate")
axis, _ = plot_polarization_ellipses_across_field(
    X, Y, Jall_m, E_in
)
_ = axis.set_title("Multi Order plate")


You can see for the multi wave plate that the ellipticity of the beam is getting worse at the edge. We can also plot the rays going through the system as a visual aid.  Here we use a high angle of incidence to make it easier to see the separation of the two polarizations.

In [ ]:
tta = 70
theta = np.deg2rad(tta)
k_in = np.array([0, np.sin(theta), np.cos(theta)])
pos_in = np.array([0, 0, 0])
E_in = np.array([0, 1])
ray_output = polarization_ray_trace(
    T, k_in, pos_in, E_in, trace_options
)

Tplot = copy.deepcopy(T)
update_clear_apertures_from_ray_trace(
    Tplot, ray_output, margin=1.15, minimum=0.25
)
axis, _ = plot_prt_lens_cross_section(Tplot)
_ = plot_prt_ray_trace(ray_output, ax=axis)


You can also do a 3D plot of this that will have polarization glphs so you can see how each surface impacts the polarization state: